# 06 — Bottleneck Analysis
**CareFlow Process Mining | Google Colab**

Calculates elapsed minutes between consecutive recorded events in each case, summarizes long intervals, and compares transition intervals. These are inter-event times, not necessarily time spent performing the activity and not proof of a clinical bottleneck.

In [ ]:
!pip -q install openpyxl pandas numpy matplotlib seaborn pm4py

import io, os, re, warnings
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import files
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)
print("CareFlow Process Mining environment ready.")

In [ ]:
# Upload: Sepsis_Cases_Event_Log_Cleaned_for_Visualization.xlsx
uploaded = files.upload()
if not uploaded:
    raise ValueError("Please upload the processed sepsis workbook.")
workbook_name = next(iter(uploaded))
workbook_bytes = uploaded[workbook_name]
xls = pd.ExcelFile(io.BytesIO(workbook_bytes))
print("Workbook:", workbook_name)
print("Sheets:", xls.sheet_names)

required_sheet = "Event Log Visual"
if required_sheet not in xls.sheet_names:
    raise ValueError(f"Expected sheet '{required_sheet}'. Available sheets: {xls.sheet_names}")

events = pd.read_excel(io.BytesIO(workbook_bytes), sheet_name="Event Log Visual")
events.columns = [str(c).strip() for c in events.columns]
print("Event Log Visual shape:", events.shape)
display(events.head())

In [ ]:
# This workbook's event-log sheet uses these verified column names:
CASE_COL = "case_id"
ACTIVITY_COL = "activity"
TIME_COL = "timestamp"
EVENT_INDEX_COL = "event_index"
ORG_COL = "org_group"
LIFECYCLE_COL = "lifecycle_transition"

# Parse timestamp safely; invalid values become NaT and are reported.
if TIME_COL in events.columns:
    events[TIME_COL] = pd.to_datetime(events[TIME_COL], errors="coerce")

print("Case column:", CASE_COL, "| present:", CASE_COL in events.columns)
print("Activity column:", ACTIVITY_COL, "| present:", ACTIVITY_COL in events.columns)
print("Timestamp column:", TIME_COL, "| present:", TIME_COL in events.columns)

## 1. Prepare timestamped events

In [ ]:
time_df = events.dropna(subset=[CASE_COL, TIME_COL]).copy()
time_df[TIME_COL] = pd.to_datetime(time_df[TIME_COL], errors="coerce")
time_df = time_df.dropna(subset=[TIME_COL]).copy()
sort_cols = [CASE_COL]
if EVENT_INDEX_COL in time_df.columns:
    sort_cols.append(EVENT_INDEX_COL)
sort_cols.append(TIME_COL)
time_df = time_df.sort_values(sort_cols, na_position="last")
print("Timestamped rows:", len(time_df))

## 2. Calculate time between consecutive events

In [ ]:
time_df["next_timestamp"] = time_df.groupby(CASE_COL)[TIME_COL].shift(-1)
time_df["next_activity"] = time_df.groupby(CASE_COL)[ACTIVITY_COL].shift(-1)
time_df["interval_minutes"] = (
    time_df["next_timestamp"] - time_df[TIME_COL]
).dt.total_seconds() / 60
intervals = time_df.dropna(subset=["interval_minutes"]).copy()
negative_count = int((intervals["interval_minutes"] < 0).sum())
print("Intervals:", len(intervals))
print("Negative intervals:", negative_count)
display(intervals["interval_minutes"].describe(percentiles=[.5, .75, .9, .95, .99]).to_frame())

## 3. Longest intervals to review

In [ ]:
cols = [CASE_COL, TIME_COL, ACTIVITY_COL, "next_timestamp", "next_activity", "interval_minutes"]
longest = intervals.sort_values("interval_minutes", ascending=False)[cols].head(25)
display(longest)

## 4. Plot inter-event interval distribution

In [ ]:
valid = intervals.loc[intervals["interval_minutes"] >= 0, "interval_minutes"]
if not valid.empty:
    upper = valid.quantile(0.99)
    plot_values = valid[valid <= upper]
    plt.figure(figsize=(10, 5))
    plt.hist(plot_values, bins=40)
    plt.title("Inter-Event Time Distribution (up to 99th percentile)")
    plt.xlabel("Minutes between consecutive events")
    plt.ylabel("Number of intervals")
    plt.tight_layout()
    plt.show()
else:
    print("No valid non-negative intervals to plot.")

## 5. Transitions with longest median intervals

In [ ]:
transition_summary = (
    intervals[intervals["interval_minutes"] >= 0]
    .groupby([ACTIVITY_COL, "next_activity"])["interval_minutes"]
    .agg(event_transitions="count", median_minutes="median", mean_minutes="mean", max_minutes="max")
    .reset_index()
    .sort_values("median_minutes", ascending=False)
)
display(transition_summary.head(25))
intervals.to_csv("inter_event_intervals.csv", index=False)
transition_summary.to_csv("transition_interval_summary.csv", index=False)
files.download("inter_event_intervals.csv")
files.download("transition_interval_summary.csv")

## Interpretation safeguards
- Verify that `timestamp` has the expected meaning and timezone.
- Long intervals are candidates for review, not automatically bottlenecks.
- Some long gaps may reflect weekends, transfer, missing records, or workflow design.
- Do not infer treatment quality or patient outcomes from elapsed time alone.